In [400]:
import numpy as np

In [401]:
data_entrada_saida = [np.array([[1],[2]]),
                      np.array([[5]])]

data_w = [np.array([[1,2],
                    [-1,3]]),
        np.array([[2,-1]])]

data_b = [np.array([[1],[-2]]),
          np.array([[1]])]

eta = 0.1


In [442]:
def ReLU(z):
    return np.maximum(0, z)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def z_a_calculation(data_W, data_B, x,funcao_ativacao="Relu"):
    a = x
    zs = []
    activations = [x]

    for W, b in zip(data_W, data_B):
        z = W @ a + b
        zs.append(z)

        if(funcao_ativacao.upper()=="RELU"):
            a = ReLU(z)
        elif(funcao_ativacao.upper()=="SIGMOID"):
            a = sigmoid(z)
        activations.append(a)

    return zs, activations

def derivada_ReLU(z):
    return (z > 0).astype(float)

def derivada_sigmoid_from_z(z):
    a = sigmoid(z)
    return a * (1 - a)

def bce_loss(y, y_hat):
    eps = 1e-9
    return -(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps))

def delta(data_W, data_Z, y, y_hat,
          funcao_ativacao="RELU",
          funcao_erro="BCE"):

    L = len(data_W)
    deltas = [None] * L

    if funcao_erro.upper() == "MSE":
        d = (y_hat - y) * 2

    elif funcao_erro.upper() == "BCE":
        d = (y_hat - y)

    else:
        raise ValueError("Loss não suportada")

    deltas[-1] = d

    for l in range(L - 2, -1, -1):

        if funcao_ativacao.upper() == "RELU":
            local_grad = derivada_ReLU(data_Z[l])

        elif funcao_ativacao.upper() == "SIGMOID":
            local_grad = derivada_sigmoid_from_z(data_Z[l])

        else:
            raise ValueError("Ativação não suportada")

        d = (data_W[l + 1].T @ d) * local_grad
        deltas[l] = d

    return deltas

def gradient_calculation (delta,a):
    gradient_w = [d @ a[i].T for i, d in enumerate(delta)]
    gradient_b = [delta[i] for i in range(len(delta))]
    return gradient_w, gradient_b
def atualizacao (grad, eta, peso):
    return [peso[i] - eta*grad[i] for i in range(len(peso))]

data_entrada_saida = [
        (np.array([[0],[0]]), np.array([[0]])),
        (np.array([[0],[1]]), np.array([[1]])),
        (np.array([[1],[0]]), np.array([[1]])),
        (np.array([[1],[1]]), np.array([[0]]))
    ]

data_w = [
    np.random.randn(2, 2) * 0.1,
    np.random.randn(1, 2) * 0.1
]
print(data_w)
data_b = [
    np.zeros((2, 1)),
    np.zeros((1, 1))
]


def algorithm(data_W, data_B, data, eta, epoca = 1):
    w_list = data_W
    b_list = data_B


    for _ in range(epoca):
        for x, y in data:

            z, a = z_a_calculation(w_list, b_list, x,funcao_ativacao="Sigmoid")
            y_hat = a[-1]
            
            deltas = delta(w_list, z, y, y_hat, funcao_ativacao="Sigmoid", funcao_erro="BCE")
            grad_w, grad_b = gradient_calculation(deltas, a)

            w_list = atualizacao(grad_w, eta, w_list)
            b_list = atualizacao(grad_b, eta, b_list)

    for x, y in data:
        z, a = z_a_calculation(w_list, b_list, x, funcao_ativacao="Sigmoid")
        y_hat = a[-1]

        print("y:", y.flatten(), "y_hat:", y_hat.flatten())
        
    return w_list,b_list

w,b = algorithm(data_w, data_b, data_entrada_saida, eta,epoca=10000)



[array([[ 0.02489453, -0.01464988],
       [-0.01877835,  0.1425967 ]]), array([[0.09107253, 0.11064202]])]
y: [0] y_hat: [0.00285157]
y: [1] y_hat: [0.99695117]
y: [1] y_hat: [0.99694629]
y: [0] y_hat: [0.00461661]
